# Homework 3

018288075 <br/>
Shravankumar Nagarajan

## Part 1 Wikipedia PageRank with PySpark
Part 1 shows how to parse a small Wikipedia XML sample and compute PageRank scores using PySpark.

## Workflow
1. Start a Spark session.
2. Parse the sample Wikipedia XML dump and build an adjacency list of links.
3. Run an iterative PageRank computation.
4. Display the ten pages with the highest PageRank scores.

In [1]:
!apt-get update -q
!apt-get install openjdk-11-jdk -q -y
!pip install pyspark==3.5.2 lxml requests

zsh:1: command not found: apt-get
zsh:1: command not found: apt-get
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 5.3 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-3.5.2-py2.py3-none-any.whl size=317812365 sha256=9c48b16955af6d3f98ecb8323148cc1426c58ca136064694b9f65b363f96e85c
  Stored in directory: /Users/hiruzen/Library/Caches/pip/wheels/cf/c0/b9/f147f4220fd1d9277d0981b88b35b26f03ad910fffd60013a6
Successfully built pyspark


In [6]:
import os
import requests
import bz2
import shutil

url = "https://dumps.wikimedia.org/enwiki/20251001/enwiki-20251001-pages-articles-multistream11.xml-p6899367p7054859.bz2"
bz2_filename = "enwiki-20251001-pages-articles-multistream11.xml-p6899367p7054859.bz2"
xml_filename = bz2_filename.rstrip(".bz2")

def download_file(url, local_filename):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            shutil.copyfileobj(r.raw, f)

print(f"Downloading {url} → {bz2_filename} …")
download_file(url, bz2_filename)
print("Download finished.")

print(f"Decompressing {bz2_filename} → {xml_filename} …")
with bz2.BZ2File(bz2_filename) as fr, open(xml_filename, 'wb') as fw:
    shutil.copyfileobj(fr, fw)
print("Decompression finished.")

print(f"Done. XML file available at: {xml_filename}")

Download finished.
Decompressing enwiki-20251001-pages-articles-multistream11.xml-p6899367p7054859.bz2 → enwiki-20251001-pages-articles-multistream11.xml-p6899367p7054859 …
Decompression finished.
Done. XML file available at: enwiki-20251001-pages-articles-multistream11.xml-p6899367p7054859


In [7]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import math, os

spark = (
    SparkSession.builder
    .appName("WikipediaPageRank")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .getOrCreate()
)
sc = spark.sparkContext
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")

Spark version: 3.5.2


In [8]:
import gzip, bz2, re, html
from xml.etree import ElementTree as ET
from pathlib import Path

data_path = Path("/content/enwiki-20251001-pages-articles-multistream11.xml-p6899367p7054859")
NS = {"mw": "http://www.mediawiki.org/xml/export-0.11/"}

LINK_RE = re.compile(r"\[\[([^\]|#]+)")
SKIP_NS = ("Category:", "File:", "Image:", "Template:", "Help:", "Portal:",
           "Draft:", "Wikipedia:", "MediaWiki:", "Talk:", "User:",
           "TimedText:", "Module:", "MOS:")

def norm_title(s: str) -> str:
    s = (s or "").strip().replace("_", " ")
    return (s[:1].upper() + s[1:]) if s else s

def is_article(title: str) -> bool:
    return title and not title.startswith(SKIP_NS)

graph = {}
all_links = set()

open_fn = (
    lambda p: bz2.open(p, mode="rt", encoding="utf-8") if p.suffix == ".bz2"
    else gzip.open(p, mode="rt", encoding="utf-8") if p.suffix == ".gz"
    else open(p, "rt", encoding="utf-8")
)

with open_fn(data_path) as f:
    for event, elem in ET.iterparse(f, events=("end",)):
        if elem.tag.endswith("page"):
            title_elem = elem.find("mw:title", NS)
            text_elem  = elem.find("mw:revision/mw:text", NS)
            title = (title_elem.text or "").strip() if title_elem is not None else ""
            if not title:
                elem.clear(); continue
            graph.setdefault(title, set())
            raw = html.unescape(text_elem.text or "")
            for match in LINK_RE.findall(raw):
                tgt = norm_title(match.split("#")[0])
                if is_article(tgt):
                    graph[title].add(tgt)
                    all_links.add(tgt)
            elem.clear()

for tgt in all_links:
    graph.setdefault(tgt, set())

graph_items = [(t, sorted(neigh)) for t, neigh in graph.items()]
print(f"Pages parsed: {len(graph_items)}")

Pages parsed: 458589


In [ ]:
target_parts = max(2, sc.defaultParallelism * 8)
links_rdd = sc.parallelize([(t, tuple(neigh)) for t, neigh in graph_items], target_parts)\
              .persist(StorageLevel.MEMORY_ONLY)

num_pages = links_rdd.count()
num_edges = links_rdd.map(lambda kv: len(kv[1])).sum()
print(f"Graph has {num_pages} pages and {int(num_edges)} directed links")

damping = 0.85
num_iterations = 10
base = (1.0 - damping) / num_pages

dangling_list = links_rdd.filter(lambda kv: len(kv[1]) == 0).keys().collect()
dangling_set = set(dangling_list)

ranks = links_rdd.mapValues(lambda _: 1.0 / num_pages).persist(StorageLevel.MEMORY_ONLY)

ckpt_dir = "/content/spark-checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
sc.setCheckpointDir(ckpt_dir)

for i in range(num_iterations):
    ranks_map = dict(ranks.collect())
    br = sc.broadcast(ranks_map)
    def emit_contribs(iter_kv):
        rmap = br.value
        for src, nbrs in iter_kv:
            if nbrs:
                share = rmap.get(src, 0.0) / len(nbrs)
                for dst in nbrs:
                    yield (dst, share)

    contribs = links_rdd.mapPartitions(emit_contribs, preservesPartitioning=False)
    agg = contribs.reduceByKey(lambda a, b: a + b, numPartitions=target_parts)
    dangling_mass = sum(ranks_map.get(n, 0.0) for n in dangling_set)
    uniform_dang = dangling_mass / num_pages
    ranks = (
        links_rdd.keys().map(lambda k: (k, 0.0))
        .leftOuterJoin(agg)
        .mapValues(lambda x: x[1] if x[1] is not None else 0.0)
        .mapValues(lambda s: base + damping * (s + uniform_dang))
        .persist(StorageLevel.MEMORY_ONLY)
    )
    br.unpersist()
    print(f"Iteration {i+1} done")
    if (i + 1) % 5 == 0 and not ranks.isCheckpointed():
        ranks.checkpoint()
        _ = ranks.count()

print("PageRank iterations complete")

Graph has 458589 pages and 707687 directed links
Iteration 1 done
Iteration 2 done
Iteration 3 done
Iteration 4 done
Iteration 5 done
Iteration 6 done
Iteration 7 done
Iteration 8 done
Iteration 9 done
Iteration 10 done
PageRank iterations complete


In [ ]:
top_pages = ranks.takeOrdered(10, key=lambda kv: -kv[1])
print("\nTop 10 Wikipedia pages by PageRank:\n")
for i, (title, score) in enumerate(top_pages, 1):
    print(f"{i:2d}. {title:<60} {score:.6f}")


Top 10 Wikipedia pages by PageRank:

 1. United States                                                0.000130
 2. List of former primary state highways in Virginia (Bristol District) 0.000097
 3. List of sovereign states                                     0.000065
 4. WP:V                                                         0.000055
 5. WP:BIO                                                       0.000048
 6. WP:NOT                                                       0.000048
 7. Texas                                                        0.000046
 8. Central European Time                                        0.000045
 9. Roh Moo-hyun                                                 0.000045
10. Central European Summer Time                                 0.000045


In [ ]:
links_rdd.unpersist()
dangling.unpersist()
spark.stop()

## Part 2 Spark Examples Logestic Regression

### 2.a Explain the code to the extent you can
1. The code implements **Logistic Regression** from scratch using **Spark RDDs** for distributed data handling and **NumPy** for efficient matrix-based computation.  

2. It reads each input line formatted as `<label> <x1> <x2> … <xD>`, where the **first value (label)** represents the class (e.g., `1`(spam) or `-1` (not spam) in the case of email spam) and the remaining **D values (`x1` to `xD`)** represent the individual feature values of a single data point. Spark reads multiple such lines in parallel, and within each partition, these lines are grouped and converted into a **NumPy matrix** of size *(number_of_rows_in_partition × (D+1))*. Each row corresponds to one sample, and each column corresponds to a specific feature, enabling **vectorized gradient computations**.

3. A random weight vector **`w`** is initialized and iteratively updated using **gradient descent** across distributed data batches to minimize the **logistic loss**.  

4. The **`gradient()`** function computes the aggregated **logistic regression gradient** for all samples in each partition, applying the **sigmoid function** `1 / (1 + exp(-z))` and assuming binary labels `{−1, +1}`.  

5. Each iteration aggregates gradients from all partitions using **`reduce(add)`**, updates the global weight vector **`w`**, and prints the **final optimized weights** after completing all iterations.  

### 2.b Running the code for sklearn breast cancer dataset

In [ ]:
import numpy as np
from typing import Iterable, List
from pyspark.sql import SparkSession
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#### Helper functions: data loading, batch processing, and gradient computation

In [ ]:
def infer_D(spark, path):
    first = spark.read.text(path).rdd.map(lambda r: r[0]).first()
    return len(np.fromstring(first.replace(',', ' '), dtype=np.float32, sep=' ')) - 1

def readPointBatch(iterator: Iterable[str], D: int) -> List[np.ndarray]:
    strs = list(iterator)
    matrix = np.zeros((len(strs), D + 1), dtype=np.float32)
    for i, s in enumerate(strs):
        matrix[i] = np.fromstring(s.replace(',', ' '), dtype=np.float32, sep=' ')
    return [matrix]

def gradient(matrix: np.ndarray, w: np.ndarray) -> np.ndarray:
    Y = matrix[:, 0]
    X = matrix[:, 1:]
    return ((1.0 / (1.0 + np.exp(-Y * X.dot(w))) - 1.0) * Y * X.T).sum(1)

def add(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    x += y
    return x

def rdd_accuracy(points_rdd, w):
    def batch_acc(m):
        Y = m[:,0]
        X = m[:,1:]
        scores = X.dot(w)
        preds = np.where(scores >= 0, 1.0, -1.0)
        return np.array([np.mean(preds == Y)], dtype=np.float32)
    num = points_rdd.map(batch_acc).collect()
    return float(np.mean(num)) if num else 0.0


def load_points(spark, path, D):
    return (spark.read.text(path).rdd
            .map(lambda r: r[0])
            .mapPartitions(lambda it: readPointBatch(it, D)))

#### Prepare train/test text files the naive script expects

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
y = np.where(y == 1, 1, -1)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
train_rows = [[float(y)] + list(map(float, X.tolist())) for X, y in zip(X_train, y_train)]
test_rows  = [[float(y)] + list(map(float, X.tolist())) for X, y in zip(X_test,  y_test)]

In [ ]:
spark = (SparkSession.builder.appName("LogisticRegression").getOrCreate())
sc = spark.sparkContext

train_rdd = sc.parallelize([" ".join(map(str, row)) for row in train_rows])
test_rdd  = sc.parallelize([" ".join(map(str, row)) for row in test_rows])

D = X.shape[1]
points_train = train_rdd.mapPartitions(lambda it: readPointBatch(it, D)).cache()
points_test  = test_rdd.mapPartitions(lambda it: readPointBatch(it, D)).cache()

rng = np.random.default_rng(0)
w = rng.uniform(-1, 1, size=D).astype(np.float32)
iterations = 50

for i in range(iterations):
    grad_sum = points_train.map(lambda m: gradient(m, w)).reduce(add)
    w -= (1.0 / max(1, i + 1)) * grad_sum

print("Weight Vector (w):", w)

train_acc = rdd_accuracy(points_train, w)
test_acc  = rdd_accuracy(points_test,  w)

print(f"Logistic Regression accuracy — train: {train_acc:.4f}, test: {test_acc:.4f}")
spark.stop()


Weight Vector (w): [-206.35559  -106.239265 -211.31169  -197.70372  -157.12369  -167.3487
 -178.96776  -228.89809  -110.157906   18.498474 -149.10838    64.29067
 -142.87234  -144.28336    38.70329   -14.981234   21.256529  -69.25741
   44.11444    75.59527  -227.14195  -129.96278  -227.71358  -208.61523
 -183.40668  -179.35684  -187.3852   -246.6922   -158.40889   -94.394516]
Logistic Regression accuracy — train: 0.9538, test: 0.9035


### Interpretation

1. Weight Vector (w): The weight vector contains one coefficient per feature, where larger positive or negative values show which features strongly influence the prediction positive weights push the model toward predicting malignant (+1), while negative weights push toward benign (−1).

2. Training Accuracy (0.9538 ≈ 95.4%): The model correctly classified about 95% of the training data, meaning it has learned the patterns in the dataset effectively and the gradient descent optimization worked well.

3. Test Accuracy (0.9035 ≈ 90.4%): The model achieved around 90% accuracy on unseen test data, showing it generalizes well with only a small performance drop from training, indicating minimal overfitting.

90% test accuracy on the Breast Cancer dataset demonstrates that the logistic regression works correctly.

## 2.c Running the code with pyspark.ml.classification.LogisticRegression

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.datasets import load_breast_cancer
import pandas as pd

In [ ]:
spark = (SparkSession.builder.appName("SparkML-LogesticRegression").getOrCreate())

bc = load_breast_cancer()
pdf = pd.DataFrame(bc.data, columns=bc.feature_names)
pdf["label"] = bc.target.astype(float)
df = spark.createDataFrame(pdf)

assembler = VectorAssembler(inputCols=bc.feature_names.tolist(), outputCol="features_raw")
df2 = assembler.transform(df)

scaler = StandardScaler(withMean=True, withStd=True, inputCol="features_raw", outputCol="features")
df3 = scaler.fit(df2).transform(df2)

train, test = df3.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=100, regParam=0.0, elasticNetParam=0.0)
model = lr.fit(train)

pred_train = model.transform(train)
pred_test  = model.transform(test)
print("Weight Vector (w):", model.coefficients)

eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

print(f"pyspark.ml LogesticRegression accuracy — train: {eval_acc.evaluate(pred_train):.4f}, test: {eval_acc.evaluate(pred_test):.4f}")

spark.stop()

Weight Vector (w): [319.6690327487924,33.84086923623507,316.53442342733973,205.89535592957805,-41.3045068933236,523.1743952376862,-473.65313179119306,-538.4440440856,129.04938992436004,-242.0766743650477,-571.2419219804071,96.0397041119053,502.27092215505473,-794.1241731724334,-126.6980757262766,-273.4813988221434,269.9882649973524,-196.17038696902398,98.53777971220168,670.357458136267,-516.1887007995342,-342.15522838351853,-177.22063987153433,-618.1546895294364,180.9345667315942,123.37684355871536,-170.24326953074947,-83.08698698693878,-140.26954239568022,-258.6941230294297]
pyspark.ml LogesticRegression accuracy — train: 1.0000, test: 0.9368


### Interpretation

1. Training Accuracy (1.0000 = 100%): The model correctly classified all training samples, showing that Spark’s ml LogesticRegression algorithm fit the training data perfectly and converged quickly.

2. Test Accuracy (0.9368 ≈ 93.7%): The model achieved about 94% accuracy on unseen data, showing excellent generalization with only a small drop from training accuracy.

The Spark ML model performs slightly better than the custom implementation (94% vs 90% test accuracy) because it uses advanced optimized LBFGS algorithm and numerical stability. Both models correctly learn the data patterns, but Spark ML’s version is faster, more accurate, and production-ready.



## Part 3 - Algorithm Streaming Integer Sum with DGIM

In [1]:
import os, subprocess

os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@11/11.0.26/libexec/openjdk.jdk/Contents/Home"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ":" + os.environ.get("PATH","")

for var in ("_JAVA_OPTIONS", "JAVA_TOOL_OPTIONS", "JENV_VERSION"):
    os.environ.pop(var, None)

extra = "--enable-native-access=ALL-UNNAMED"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'--conf spark.driver.extraJavaOptions="{extra}" '
    f'--conf spark.executor.extraJavaOptions="{extra}" pyspark-shell'
)

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print(subprocess.check_output(["java","-version"], stderr=subprocess.STDOUT).decode())

JAVA_HOME = /opt/homebrew/Cellar/openjdk@11/11.0.26/libexec/openjdk.jdk/Contents/Home
openjdk version "11.0.26" 2025-01-21
OpenJDK Runtime Environment Homebrew (build 11.0.26+0)
OpenJDK 64-Bit Server VM Homebrew (build 11.0.26+0, mixed mode)



In [2]:
!export JAVA_HOME=/opt/homebrew/Cellar/openjdk@11/11.0.26/libexec/openjdk.jdk/Contents/Home
!export PATH="$JAVA_HOME/bin:$PATH"
!python -c 'from pyspark.sql import SparkSession; print(SparkSession.builder.master("local[*]").getOrCreate()._jvm.java.lang.System.getProperty("java.version"))'

25/10/21 22:33:49 WARN Utils: Your hostname, Shravankumars-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.91 instead (on interface en0)
25/10/21 22:33:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/21 22:33:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
11.0.26


In [1]:
from pyspark.sql import SparkSession, functions as F
from collections import deque

class DGIMBit:
    def __init__(self, R=2):
        self.R = R; self.buckets = []; self.t = 0
    def tick(self): self.t += 1
    def add_one(self):
        self.buckets.insert(0, (self.t, 1)); self._merge()
    def _merge(self):
        i = 0
        while i < len(self.buckets) - 1:
            size = self.buckets[i][1]
            same = [j for j in range(len(self.buckets)) if self.buckets[j][1] == size]
            if len(same) > self.R:
                j1, j2 = same[-2], same[-1]
                t_new = self.buckets[j1][0]
                del self.buckets[j2]; del self.buckets[j1]
                self.buckets.insert(j1, (t_new, size * 2))
                return self._merge()
            i += 1
    def estimate_last_k(self, k):
        cutoff = self.t - k; total = 0; oldest = None
        for (t, size) in self.buckets:
            if t <= cutoff: break
            total += size; oldest = (t, size)
        if oldest: total -= oldest[1] // 2
        return total

class DGIMInteger:
    def __init__(self, bits=8, R=2):
        self.bits = bits; self.layers = [DGIMBit(R) for _ in range(bits)]
    def insert(self, x):
        for layer in self.layers: layer.tick()
        for b in range(self.bits):
            if (x >> b) & 1: self.layers[b].add_one()
    def query_sum(self, k):
        return sum((1 << b) * layer.estimate_last_k(k) for b, layer in enumerate(self.layers))

In [ ]:
spark = (
    SparkSession.builder
    .appName("DGIM-Integer-Kafka")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.2,"
            "org.apache.kafka:kafka-clients:3.5.2")
    .getOrCreate()
)

dgim = DGIMInteger(bits=8)
K = 50
seen = 0

def process(df, epoch):
    global seen
    # cast Kafka value (binary) -> string column
    vals = df.selectExpr("CAST(value AS STRING) AS value")
    # stream rows to driver without collecting the whole batch in memory
    for r in vals.toLocalIterator():
        try:
            x = int(r["value"])
        except Exception:
            continue
        dgim.insert(x); seen += 1
        if seen >= K and seen % 100 == 0:  # print occasionally
            print(f"Sum of last {K} numbers ≈ {dgim.query_sum(K)}")

df = (spark.readStream
          .format("kafka")
          .option("kafka.bootstrap.servers", "localhost:9092")
          .option("subscribe", "ints")
          .option("startingOffsets", "latest")
          .load())

(df.writeStream
   .outputMode("update")
   .foreachBatch(process)
   .start()
   .awaitTermination())

25/10/22 01:01:04 WARN Utils: Your hostname, Shravankumars-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.91 instead (on interface en0)
25/10/22 01:01:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/hiruzen/.ivy2/cache
The jars for the packages stored in: /Users/hiruzen/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b2e59a03-466e-459f-8359-1225f9572b72;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/hiruzen/miniconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.2 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.apache.kafka#kafka-clients;3.5.2 in central
	found com.github.luben#zstd-jni;1.5.5-1 in central
	found org.lz4#lz4-java;1.8.0 in central
:: resolution report :: resolve 224ms :: artifacts dl 5ms
	:: modules in use:
	com.github.luben#zstd-jni;1.5.5-1 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 fr

Sum of last 50 numbers ≈ 2505
Sum of last 50 numbers ≈ 2828
Sum of last 50 numbers ≈ 2196
Sum of last 50 numbers ≈ 2690
Sum of last 50 numbers ≈ 2350
Sum of last 50 numbers ≈ 2384
Sum of last 50 numbers ≈ 3230
Sum of last 50 numbers ≈ 2820
Sum of last 50 numbers ≈ 2084


In [ ]:
spark.stop()